# Prediction of Real State Prices in Portugal

The current project involves preparing the necessary data and training a supervised model to predict the price of a real estate property based on various variables. 

We will use the LightGBM algorithm, which is an optimized implementation of gradient boosting. This algorithm is based on an ensemble of decision trees that are trained sequentially, with each tree learning to correct the errors of the previous ones. LightGBM is particularly effective due to its ability to handle large datasets, work with sparse features, and capture complex patterns in the data efficiently.


# Table of contents

- [Libraries and functions](#libraries-and-functions)
- [Loading and initial approach to data](#loading-and-initial-approach-to-data)
- [Imputation of nulls](#imputation-of-nulls)
- [Encoding qualitative variables](#encoding-qualitative-variables)
- [Outliers](#outliers)
- [Preparing the dataset for the model](#preparing-the-dataset-for-the-model)
- [Training the model](#training-the-model)
- [Evaluating the model](#evaluating-the-model)
- [Splitting the dataset into three ranges](#splitting-the-dataset-into-three-ranges)
- [Conclusions](#conclusions)

## Libraries and functions

In [240]:
import pandas as pd
import polars as pl
import numpy as np
from datetime import datetime
import re
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from lightgbm.callback import early_stopping
from lightgbm import log_evaluation
from sklearn.preprocessing import StandardScaler

In [241]:
def poor_columns(df):
    threshold = 0.5 * df.height
    poor_columns = [c for c in df.columns if df.select(pl.col(c).is_null().sum()).item() > threshold]
    return display(poor_columns)

In [242]:
def codificar_floor(floor):
    floor = str(floor).lower() 
    if floor == 'no':  
        return 0 
    elif "basement level" in floor: 
        return -1
    elif "ground floor" in floor:  
        return 0
    elif "mezzanine" in floor:  
        return 0.5
    elif "top floor" in floor:  
        return 15
    elif "attic" in floor:  
        return 15
    elif "above 10th floor" in floor:  
        return 15  
    elif "service floor" in floor:  
        return 1
    elif "triplex" in floor:  
        return 15
    elif "duplex" in floor:  
        return 15
    else:
        match = re.search(r"(\d+)", floor)
        return float(match.group(1)) if match else None

In [243]:
def codificar_energy_certificate(cert):
    if cert is None or cert.lower() in ["no certificate", "not available", "nc"]:
        return -1
    cert = cert.upper() 
    mapping = {
        "A+": 1,
        "A": 2,
        "B": 3,
        "B-": 4,
        "C": 5,
        "D": 6,
        "E": 7,
        "F": 8,
        "G": 9
    }
    return mapping.get(cert, -1) 


## Loading and initial approach to data

In [244]:
df_house = pl.read_csv('portugal_housing.csv')
df_house.sample(5)

Price,District,City,Town,Type,EnergyCertificate,Floor,Lift,Parking,HasParking,ConstructionYear,TotalArea,GrossArea,PublishDate,Garage,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea,NumberOfBathrooms
f64,str,str,str,str,str,str,bool,f64,bool,f64,f64,f64,str,str,str,str,f64,str,str,str,f64,str,str,f64
125000.0,"""Faro""","""Lagos""","""São Gonçalo de Lagos""","""Garage""","""NC""",null,null,0.0,null,2004.0,80.0,null,"""2024-08-29 12:34:02.980""","""True""","""False""","""False""",null,null,null,null,null,null,null,null
319900.0,"""Porto""","""Gondomar""","""Rio Tinto""","""House""","""D""",null,null,0.0,null,1981.0,304.0,null,null,"""True""","""False""","""True""",null,"""4.0""","""1.0""","""Good condition""",196.0,"""283.0""",null,3.0
945000.0,"""Faro""","""Vila do Bispo""","""Vila do Bispo e Raposeira""","""Other - Commercial""","""B-""",null,null,1.0,null,2008.0,434.0,null,null,"""False""","""False""","""False""",null,"""3.0""","""5.0""",null,468.0,"""1221.0""",null,7.0
115000.0,"""Aveiro""","""Ovar""","""Ovar, São João, Arada e São Vi…","""Land""","""NC""",null,null,0.0,null,1972.0,null,null,null,"""False""","""False""","""False""",null,null,"""0.0""",null,null,"""1527.0""",null,0.0
398500.0,"""Bragança""","""Bragança""","""Santa Comba de Rossas""","""House""","""C""",null,false,0.0,false,1938.0,1935.0,null,null,null,null,null,9.0,null,null,null,840.0,null,null,6.0


In [245]:
# We detect the columns called poor as they do not have enough instances with data
poor_columns = poor_columns(df_house)

['Floor',
 'GrossArea',
 'PublishDate',
 'Garage',
 'Elevator',
 'ElectricCarsCharging',
 'NumberOfBedrooms',
 'NumberOfWC',
 'ConservationStatus',
 'LotSize',
 'BuiltArea']

In [246]:
# We select the columns that we need 
df_house = df_house.select(['Price','District','City','Town','Type','EnergyCertificate','Floor','Lift','Parking',
                            'ConstructionYear','TotalArea','LivingArea','TotalRooms','NumberOfBathrooms'])

In [247]:
# We drop the rows with null price 
df_house = df_house.filter(pl.col("Price").is_not_null())

# We convert the category of some columns and rename
df_house = df_house.with_columns([
    pl.col('Parking').cast(pl.Int64),
    pl.col('ConstructionYear').cast(pl.Int64),
    pl.col('TotalRooms').cast(pl.Int64),
    pl.col('NumberOfBathrooms').cast(pl.Int64)
])

df_house = df_house.rename({
    'Parking': 'No.Parkings',
    'NumberOfBathrooms': 'No.Bathrooms'
})

## Imputation of nulls

In [248]:
# We asume that a null value on 'Lift' means that there is no lift
# When null value appears in 'Floor' it means the property has no floor
# We impute the null values on other variables in a coherent way
# We also impute the value of the 'TotalArea' to the 'LivingArea' when it is missing.

df_house = df_house.with_columns(
            pl.col('Lift').fill_null(False), 
            pl.col('Floor').fill_null('no'),
            pl.col('No.Parkings').fill_null(0), 
            pl.col("TotalArea").fill_null(5),
            pl.col("TotalRooms").fill_null(0),
            pl.col("No.Bathrooms").fill_null(0)
)

df_house = df_house.with_columns(
    pl.when(pl.col('LivingArea').is_null())
    .then(pl.col('TotalArea'))  
    .otherwise(pl.col('LivingArea')) 
    .alias('LivingArea')
)

In [249]:
df_house.sample(2)

Price,District,City,Town,Type,EnergyCertificate,Floor,Lift,No.Parkings,ConstructionYear,TotalArea,LivingArea,TotalRooms,No.Bathrooms
f64,str,str,str,str,str,str,bool,i64,i64,f64,f64,i64,i64
189000.0,"""Aveiro""","""São João da Madeira""","""São João da Madeira""","""Apartment""","""E""","""no""",false,1,2000,142.0,142.0,8,3
325000.0,"""Porto""","""Gondomar""","""Baguim do Monte""","""Apartment""","""D""","""no""",false,0,2000,196.0,196.0,0,2


## Encoding qualitative variables

Now, we encode all qualitative variables in order to being able to train the model.

In [250]:
current_year = datetime.now().year

df_house = df_house.with_columns([
    pl.col('District').cast(pl.Categorical).to_physical().alias('District_encoded'),
    pl.col('City').cast(pl.Categorical).to_physical().alias('City_encoded'),
    pl.col('Town').cast(pl.Categorical).to_physical().alias('Town_encoded'),
    pl.col('Type').cast(pl.Categorical).to_physical().alias('Type_encoded'),
    pl.col('Lift').cast(pl.Int32).alias('Lift_encoded'),
    (current_year - pl.col('ConstructionYear')).alias('Property_Age')
])

floor_encoded = [codificar_floor(floor) for floor in df_house['Floor']]
df_house = df_house.with_columns(pl.Series('Floor_encoded', floor_encoded))

energy_encoded = [codificar_energy_certificate(cert) for cert in df_house['EnergyCertificate']]
df_house = df_house.with_columns(pl.Series('EnergyCertificate_encoded', energy_encoded))

# As we have many nulls on the ConstructionYear columns and this implies several null values in Property_Age, we impute them with the mean age
df_house = df_house.with_columns(pl.col('Property_Age').fill_null(round(df_house['Property_Age'].mean(),0)))


In [251]:
df_house.sample(3)

Price,District,City,Town,Type,EnergyCertificate,Floor,Lift,No.Parkings,ConstructionYear,TotalArea,LivingArea,TotalRooms,No.Bathrooms,District_encoded,City_encoded,Town_encoded,Type_encoded,Lift_encoded,Property_Age,Floor_encoded,EnergyCertificate_encoded
f64,str,str,str,str,str,str,bool,i64,i64,f64,f64,i64,i64,u32,u32,u32,u32,i32,f64,f64,i64
235000.0,"""Santarém""","""Torres Novas""","""Torres Novas (Santa Maria, Sal…","""Apartment""","""C""","""2nd Floor""",false,1,2001,153.0,128.0,3,2,12,219,1580,0,0,24.0,2.0,5
160000.0,"""Santarém""","""Tomar""","""Carregueiros""","""House""","""E""","""no""",false,2,1981,444.0,75.0,3,2,12,64,1570,1,0,44.0,0.0,7
379000.0,"""Porto""","""Porto""","""Cedofeita, Santo Ildefonso, Sé…","""Store""","""C""","""no""",false,0,null,69.0,37.0,0,0,3,17,1673,6,0,36.0,0.0,5


## Outliers
Here we filter out the anomalous values that do not make apparent sense.

In [252]:
df_house = df_house.filter(
    (pl.col('Price') <= 100_000_000) &  # Reasonable prices
    (pl.col('Town').is_not_null()) &
    (pl.col('Type').is_not_null()) &
    (pl.col('TotalArea') >= 5) & (pl.col('TotalArea') <= 200_000_000) &  # Reasonable Total Area (m2)
    (pl.col('LivingArea') >= 5) & (pl.col('LivingArea') <= 200_000_000) &  # Reasonable Living Area (m2)
    (pl.col('TotalRooms') <= 100) &  # Reasonable Number of rooms
    (pl.col('No.Bathrooms') >= 0) # Drop negative values
)

In [253]:
df_house.describe()

statistic,Price,District,City,Town,Type,EnergyCertificate,Floor,Lift,No.Parkings,ConstructionYear,TotalArea,LivingArea,TotalRooms,No.Bathrooms,District_encoded,City_encoded,Town_encoded,Type_encoded,Lift_encoded,Property_Age,Floor_encoded,EnergyCertificate_encoded
str,f64,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",113102.0,"""113102""","""113102""","""113102""","""113102""","""113102""","""113102""",113102.0,113102.0,72642.0,113102.0,113102.0,113102.0,113102.0,113102.0,113102.0,113102.0,113102.0,113102.0,113102.0,113016.0,113102.0
"""null_count""",0.0,"""0""","""0""","""0""","""0""","""0""","""0""",0.0,0.0,40460.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,86.0,0.0
"""mean""",353141.115238,null,null,null,null,null,null,0.147486,0.582934,1988.819608,15149.754522,6690.509558,1.745937,1.432406,7.044341,108.265566,904.593102,2.058239,0.147486,36.11586,0.517829,2.249775
"""std""",631389.560841,null,null,null,null,null,null,null,0.883741,26.724336,1.0109e6,591113.807275,2.463621,1.697764,5.30899,75.042999,634.472797,2.671456,0.354592,21.417469,1.61516,3.365157
"""min""",100.0,"""Aveiro""","""Abrantes""","""A dos Cunhados e Maceira""","""Apartment""","""A""","""1st Floor""",0.0,0.0,1900.0,5.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-1.0,-1.0
"""25%""",77500.0,null,null,null,null,null,null,null,0.0,1972.0,87.0,79.0,0.0,0.0,3.0,41.0,435.0,0.0,0.0,25.0,0.0,-1.0
"""50%""",200000.0,null,null,null,null,null,null,null,0.0,1994.0,162.0,132.0,1.0,1.0,4.0,95.0,701.0,1.0,0.0,36.0,0.0,2.0
"""75%""",390000.0,null,null,null,null,null,null,null,1.0,2008.0,560.0,309.0,3.0,2.0,12.0,174.0,1550.0,3.0,0.0,38.0,0.0,6.0
"""max""",3.6e7,"""Évora""","""Óbidos""","""Évora de Alcobaça""","""Warehouse""","""Not available""","""no""",1.0,3.0,2024.0,1.656206e8,1.656206e8,97.0,90.0,24.0,271.0,2236.0,20.0,1.0,125.0,15.0,9.0


In [254]:
# We verify that LivingArea and TotalArea are not redundant
corr = df_house[["LivingArea", "TotalArea"]].corr()
print(corr)

shape: (2, 2)
┌────────────┬───────────┐
│ LivingArea ┆ TotalArea │
│ ---        ┆ ---       │
│ f64        ┆ f64       │
╞════════════╪═══════════╡
│ 1.0        ┆ 0.584311  │
│ 0.584311   ┆ 1.0       │
└────────────┴───────────┘


## Preparing the dataset for the model
We transform the target column (price) to a logarithmic scale so that the model can better capture the distributions. After that, we create the dataset by selecting from the original one the variables that the model will use.

In [255]:
c = 1000
df_house = df_house.with_columns(
    np.log1p(df_house['Price'] + c).alias('Price_log')
)

In [256]:
df_house_clean = df_house.select(['Price','Price_log','District_encoded','City_encoded','Town_encoded','Type_encoded',
                                  'EnergyCertificate_encoded','Floor_encoded','Property_Age','Lift_encoded',
                                  'TotalArea','LivingArea','TotalRooms','No.Bathrooms','No.Parkings'])

In [257]:
df_house_clean.describe()

statistic,Price,Price_log,District_encoded,City_encoded,Town_encoded,Type_encoded,EnergyCertificate_encoded,Floor_encoded,Property_Age,Lift_encoded,TotalArea,LivingArea,TotalRooms,No.Bathrooms,No.Parkings
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",113102.0,113102.0,113102.0,113102.0,113102.0,113102.0,113102.0,113016.0,113102.0,113102.0,113102.0,113102.0,113102.0,113102.0,113102.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,86.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",353141.115238,12.080694,7.044341,108.265566,904.593102,2.058239,2.249775,0.517829,36.11586,0.147486,15149.754522,6690.509558,1.745937,1.432406,0.582934
"""std""",631389.560841,1.23774,5.30899,75.042999,634.472797,2.671456,3.365157,1.61516,21.417469,0.354592,1.0109e6,591113.807275,2.463621,1.697764,0.883741
"""min""",100.0,7.003974,0.0,0.0,0.0,0.0,-1.0,-1.0,1.0,0.0,5.0,5.0,0.0,0.0,0.0
"""25%""",77500.0,11.270867,3.0,41.0,435.0,0.0,-1.0,0.0,25.0,0.0,87.0,79.0,0.0,0.0,0.0
"""50%""",200000.0,12.211065,4.0,95.0,701.0,1.0,2.0,0.0,36.0,0.0,162.0,132.0,1.0,1.0,0.0
"""75%""",390000.0,12.876465,12.0,174.0,1550.0,3.0,6.0,0.0,38.0,0.0,560.0,309.0,3.0,2.0,1.0
"""max""",3.6e7,17.399057,24.0,271.0,2236.0,20.0,9.0,15.0,125.0,1.0,1.656206e8,1.656206e8,97.0,90.0,3.0


## Training the model

We split the data into 3 groups: **training (60%)**, **validation (20%)** and **test (20%)**.

In [258]:
X = df_house_clean.drop(['Price_log','Price'])
y = df_house_clean['Price_log']

# First we split into training and test with a 80/20 proportion
X_train_full, X_test, y_train_full, y_test = train_test_split(X.to_pandas(), y.to_pandas(), test_size=0.2, random_state=42)

# After that we split the training data into training and validation with a 75/25 proportion
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=42)

In [259]:
model = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=10, random_state=42)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric='rmse', callbacks=[early_stopping(stopping_rounds=50), log_evaluation(period=10)])

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002856 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1228
[LightGBM] [Info] Number of data points in the train set: 67860, number of used features: 13
[LightGBM] [Info] Start training from score 12.081381
Training until validation scores don't improve for 50 rounds
[10]	valid_0's rmse: 1.00169	valid_0's l2: 1.00338
[20]	valid_0's rmse: 0.883392	valid_0's l2: 0.780381
[30]	valid_0's rmse: 0.817738	valid_0's l2: 0.668695
[40]	valid_0's rmse: 0.776607	valid_0's l2: 0.603119
[50]	valid_0's rmse: 0.747965	valid_0's l2: 0.559451
[60]	valid_0's rmse: 0.726936	valid_0's l2: 0.528436
[70]	valid_0's rmse: 0.711537	valid_0's l2: 0.506285
[80]	valid_0's rmse: 0.700426	valid_0's l2: 0.490597
[90]	valid_0's rmse: 0.691303	valid_0's l2: 0.4779
[100]	valid_0's rmse: 0.685388	valid_0's l2: 0.469757
[110]

LGBMRegressor(learning_rate=0.05, max_depth=10, n_estimators=1000,
              random_state=42)

## Evaluating the model

We evaluate how good is the model on predicting the price of a property.

In [260]:
y_pred = model.predict(X_test)
y_pred_original = np.expm1(y_pred) - c
y_test_original = np.expm1(y_test) - c

mae_original = mean_absolute_error(y_test_original, y_pred_original)
mse_original = mean_squared_error(y_test_original, y_pred_original)
rmse_original = np.sqrt(mse_original)
r2_original = r2_score(y_test_original, y_pred_original)

print(f"Mean Absolute Error (Original Scale): {mae_original:.4f}")
print(f"Mean Squared Error (Original Scale): {mse_original:.4f}")
print(f"Root Mean Squared Error (Original Scale): {rmse_original:.4f}")
print(f"R-squared (Original Scale): {r2_original:.4f}")

Mean Absolute Error (Original Scale): 122815.7026
Mean Squared Error (Original Scale): 178247466922.7449
Root Mean Squared Error (Original Scale): 422193.6368
R-squared (Original Scale): 0.5291


As we can see, the model's evaluation metrics are somewhat poor. An RMSE value of around €400,000 is not ideal for predicting property prices. This is due to the wide range of prices, which span from €100 to €36M, making it difficult for the model to accurately capture the distribution. Consequently, we will implement improvement measures to refine the prediction.

## Splitting the dataset into three ranges

As we have seen, the model is not entirely effective due to the wide range of prices and their heterogeneous distribution (with more prices in the lower range than in the higher range). Therefore, we decided to split the dataset into three groups: low prices, high prices, and ultra-high prices. We will train a model for each group and compare the results with the previous model to determine if we have achieved any improvement.

In [261]:
df_low = df_house_clean.filter(pl.col('Price') <= 1_000_000)
df_high = df_house_clean.filter(pl.col('Price').is_between(1_000_000, 5_000_000, closed='right'))
df_uhigh = df_house_clean.filter(pl.col('Price') > 5_000_000)

# Low prices
X_low = df_low.drop(['Price_log','Price'])
y_low = df_low['Price_log']

X_train_full_low, X_test_low, y_train_full_low, y_test_low = train_test_split(X_low.to_pandas(), y_low.to_pandas(), test_size=0.2, random_state=42)
X_train_low, X_val_low, y_train_low, y_val_low = train_test_split(X_train_full_low, y_train_full_low, test_size=0.25, random_state=42)

# High prices
X_high = df_high.drop(['Price_log','Price'])
y_high = df_high['Price_log']

X_train_full_high, X_test_high, y_train_full_high, y_test_high = train_test_split(X_high.to_pandas(), y_high.to_pandas(), test_size=0.2, random_state=42)
X_train_high, X_val_high, y_train_high, y_val_high = train_test_split(X_train_full_high, y_train_full_high, test_size=0.25, random_state=42)

# Ultra high prices
X_uhigh = df_uhigh.drop(['Price_log','Price'])
y_uhigh = df_uhigh['Price_log']

X_train_full_uhigh, X_test_uhigh, y_train_full_uhigh, y_test_uhigh = train_test_split(X_uhigh.to_pandas(), y_uhigh.to_pandas(), test_size=0.2, random_state=42)
X_train_uhigh, X_val_uhigh, y_train_uhigh, y_val_uhigh = train_test_split(X_train_full_uhigh, y_train_full_uhigh, test_size=0.25, random_state=42)

### Low prices

In [262]:
model_low = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=10, random_state=42)
model_low.fit(X_train_low, y_train_low, eval_set=[(X_val_low, y_val_low)], eval_metric='rmse', callbacks=[early_stopping(stopping_rounds=50), log_evaluation(period=10)])

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001331 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1208
[LightGBM] [Info] Number of data points in the train set: 63951, number of used features: 13
[LightGBM] [Info] Start training from score 11.936602
Training until validation scores don't improve for 50 rounds
[10]	valid_0's rmse: 0.903362	valid_0's l2: 0.816063
[20]	valid_0's rmse: 0.796307	valid_0's l2: 0.634105
[30]	valid_0's rmse: 0.737035	valid_0's l2: 0.543221
[40]	valid_0's rmse: 0.702405	valid_0's l2: 0.493373
[50]	valid_0's rmse: 0.678585	valid_0's l2: 0.460478
[60]	valid_0's rmse: 0.662375	valid_0's l2: 0.438741
[70]	valid_0's rmse: 0.649284	valid_0's l2: 0.42157
[80]	valid_0's rmse: 0.639066	valid_0's l2: 0.408406
[90]	valid_0's rmse: 0.630996	valid_0's l2: 0.398156
[100]	valid_0's rmse: 0.625032	valid_0's l2: 0.390666
[1

LGBMRegressor(learning_rate=0.05, max_depth=10, n_estimators=1000,
              random_state=42)

In [263]:
y_pred_low = model_low.predict(X_test_low)
y_pred_original_low = np.expm1(y_pred_low) - c
y_test_original_low = np.expm1(y_test_low) - c

mae_original_low = mean_absolute_error(y_test_original_low, y_pred_original_low)
mse_original_low = mean_squared_error(y_test_original_low, y_pred_original_low)
rmse_original_low = np.sqrt(mse_original_low)
r2_original_low = r2_score(y_test_original_low, y_pred_original_low)

print(f"Mean Absolute Error (Original Scale): {mae_original_low:.4f}")
print(f"Mean Squared Error (Original Scale): {mse_original_low:.4f}")
print(f"Root Mean Squared Error (Original Scale): {rmse_original_low:.4f}")
print(f"R-squared (Original Scale): {r2_original_low:.4f}")

Mean Absolute Error (Original Scale): 65394.1374
Mean Squared Error (Original Scale): 13268569722.3259
Root Mean Squared Error (Original Scale): 115189.2778
R-squared (Original Scale): 0.7192


### High prices

In [264]:
model_high = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=10, random_state=42)
model_high.fit(X_train_high, y_train_high, eval_set=[(X_val_high, y_val_high)], eval_metric='rmse', callbacks=[early_stopping(stopping_rounds=50), log_evaluation(period=10)])

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000409 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1122
[LightGBM] [Info] Number of data points in the train set: 3756, number of used features: 13
[LightGBM] [Info] Start training from score 14.352916
Training until validation scores don't improve for 50 rounds
[10]	valid_0's rmse: 0.334249	valid_0's l2: 0.111723
[20]	valid_0's rmse: 0.320532	valid_0's l2: 0.102741
[30]	valid_0's rmse: 0.311156	valid_0's l2: 0.0968181
[40]	valid_0's rmse: 0.30561	valid_0's l2: 0.0933977
[50]	valid_0's rmse: 0.30161	valid_0's l2: 0.0909684
[60]	valid_0's rmse: 0.298879	valid_0's l2: 0.0893284
[70]	valid_0's rmse: 0.296226	valid_0's l2: 0.0877499
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[80]	valid_0's rmse: 0.29458	valid_0's l2: 0.0867772
[LightGBM] [Warning] No further

LGBMRegressor(learning_rate=0.05, max_depth=10, n_estimators=1000,
              random_state=42)

In [265]:
y_pred_high = model_high.predict(X_test_high)
y_pred_original_high = np.expm1(y_pred_high) - c
y_test_original_high = np.expm1(y_test_high) - c

mae_original_high = mean_absolute_error(y_test_original_high, y_pred_original_high)
mse_original_high = mean_squared_error(y_test_original_high, y_pred_original_high)
rmse_original_high = np.sqrt(mse_original_high)
r2_original_high = r2_score(y_test_original_high, y_pred_original_high)

print(f"Mean Absolute Error (Original Scale): {mae_original_high:.4f}")
print(f"Mean Squared Error (Original Scale): {mse_original_high:.4f}")
print(f"Root Mean Squared Error (Original Scale): {rmse_original_high:.4f}")
print(f"R-squared (Original Scale): {r2_original_high:.4f}")

Mean Absolute Error (Original Scale): 426315.9572
Mean Squared Error (Original Scale): 403847185840.2635
Root Mean Squared Error (Original Scale): 635489.7213
R-squared (Original Scale): 0.3964


### Ultra-high prices

In [266]:
model_uhigh = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=10, random_state=42)
model_uhigh.fit(X_train_uhigh, y_train_uhigh, eval_set=[(X_val_uhigh, y_val_uhigh)], eval_metric='rmse', callbacks=[early_stopping(stopping_rounds=50), log_evaluation(period=10)])

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000579 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 234
[LightGBM] [Info] Number of data points in the train set: 151, number of used features: 11
[LightGBM] [Info] Start training from score 15.878052
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM]

LGBMRegressor(learning_rate=0.05, max_depth=10, n_estimators=1000,
              random_state=42)

In [267]:
y_pred_uhigh = model_uhigh.predict(X_test_uhigh)
y_pred_original_uhigh = np.expm1(y_pred_uhigh) - c
y_test_original_uhigh = np.expm1(y_test_uhigh) - c

mae_original_uhigh = mean_absolute_error(y_test_original_uhigh, y_pred_original_uhigh)
mse_original_uhigh = mean_squared_error(y_test_original_uhigh, y_pred_original_uhigh)
rmse_original_uhigh = np.sqrt(mse_original_uhigh)
r2_original_uhigh = r2_score(y_test_original_uhigh, y_pred_original_uhigh)

print(f"Mean Absolute Error (Original Scale): {mae_original_uhigh:.4f}")
print(f"Mean Squared Error (Original Scale): {mse_original_uhigh:.4f}")
print(f"Root Mean Squared Error (Original Scale): {rmse_original_uhigh:.4f}")
print(f"R-squared (Original Scale): {r2_original_uhigh:.4f}")

Mean Absolute Error (Original Scale): 2728542.7210
Mean Squared Error (Original Scale): 23543852018079.0195
Root Mean Squared Error (Original Scale): 4852200.7397
R-squared (Original Scale): 0.0274


## Conclusions

After analyzing the results obtained, it can be concluded that dividing the data into three ranges and training a model for each range provides some improvement in predictions, especially significantly in the low-price range. It is true that in the ultra-expensive price range, the model does not fit well and the predictions are somewhat poor, but this is due to the greater dispersion of the data in this range. Thanks to the division, we avoid this high dispersion in the high-price range affecting the prediction of prices in other ranges.

In conclusion, we have obtained a good model for the low-price range and a somewhat mediocre one for the high-price range. The model for ultra-high prices is not useful. Regarding the choice of the LightGBM algorithm, it has been selected for its flexibility with variables and its ability to capture patterns in property features. Given the difficulty of the predictions, we cannot say it has been a poor choice, and if better performance is desired, we would need to use more sophisticated algorithms such as neural networks.